# E15B CPU harness (Kaggle pinned image)

The B unit tests + the fake-server harness of both E15B notebooks (45K30K: commit, rerun, fault kill / kill2 / slow; 40K35K: commit, rerun) on the same image as the GPU notebook. Kit: `foysalemonshanto/q27-e15b-cpu-kit` (q27x_e15b_cpu_kit_56ace8278542.bin).


In [ ]:
# E15B CPU harness on Kaggle (pinned image = the GPU notebook's): B unit tests + the fake-server harness of both notebooks
import hashlib, io, json, os, shutil, subprocess, sys, time, zipfile
from pathlib import Path
KIT_NAME, KIT_SHA = "q27x_e15b_cpu_kit_56ace8278542.bin", "56ace827854241c1d14756dc743624a4963f1d10c5276984e1672a0a13731719"
NBS = {"j1": "Q27-XHIGH-E15B_EVAL120_50K45K_S16_R3K1_DUCK_SGLANG.ipynb", "j2": "Q27-XHIGH-E15B_EVAL120_50K45K_S16_R3K2_DUCK_SGLANG.ipynb"}
RUNS = [["j1_eval120", "j1", "eval120", ""], ["j1_rerun", "j1", "rerun", ""], ["j2_eval120", "j2", "eval120", ""], ["j2_rerun", "j2", "rerun", ""]]
DUCK_BIN = "nb/Build/payload/q27x_e15b_duck_9e4b89285136.bin"
WH = "/kaggle/input/datasets/mirzamilanfarabi/arc3-vllm-h100-wheelhouse-v3"
OUT = Path("/kaggle/working"); K = Path("/tmp/e15b_kit"); T0 = time.time()
REPORT = {"kit": KIT_NAME, "python": sys.version, "nb_sha": {}, "unit": None, "harness": {}}


def save():
    (OUT / "cpu_harness_report.json").write_text(json.dumps(REPORT, indent=1), encoding="utf-8")


def find(name, sha):
    for pat in (name, "*/" + name, "*/*/" + name, "*/*/*/" + name, "*/*/*/*/" + name):
        for f in sorted(Path("/kaggle/input").glob(pat)):
            b = f.read_bytes()
            if hashlib.sha256(b).hexdigest() == sha:
                return b
    raise FileNotFoundError(f"{name} ({sha[:12]}) not under /kaggle/input: attach foysalemonshanto/q27-e15b-cpu-kit")


shutil.rmtree(K, ignore_errors=True)
with zipfile.ZipFile(io.BytesIO(find(KIT_NAME, KIT_SHA))) as z:
    z.extractall(K)
man = json.loads((K / "MANIFEST.json").read_text())
bad = [k for k, h in man.items() if hashlib.sha256((K / k).read_bytes()).hexdigest() != h]
assert not bad, bad
REPORT["nb_sha"] = {k: hashlib.sha256((K / "nb" / v).read_bytes()).hexdigest() for k, v in NBS.items()}
print("kit OK:", len(man), "files |", sys.version.split()[0], "|", REPORT["nb_sha"], flush=True)
# the GPU notebook's runtime pins (its CELL 1): transformers 4.57.6 + huggingface-hub 0.36.2 from the S49 wheelhouse
r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-index", "--find-links", WH,
                    "transformers==4.57.6", "huggingface-hub==0.36.2"], capture_output=True, text=True)
REPORT["pins"] = {"rc": r.returncode, "tail": (r.stdout + r.stderr)[-800:]}
print("pins:", r.returncode, flush=True)
PKG = Path("/tmp/e15b_duck_pkg"); shutil.rmtree(PKG, ignore_errors=True)
with zipfile.ZipFile(K / DUCK_BIN) as z:          # the B Duck package exactly as the notebooks load it
    z.extractall(PKG)
ENV = dict(os.environ, E14_SGL_SRC=str(K / "sgl_ref"), E14_FIXTURES_DIR=str(K / "fixtures"), PYTHONIOENCODING="utf-8",
           E14_HARNESS_DIR=str(K / "e14src/harness"), DUCK_B_PKG=str(PKG),
           E15B_TASKS=str(K / "fixtures/data/arc-agi_evaluation_challenges.json"))
t = time.time()
r = subprocess.run([sys.executable, str(K / "e15/test_duck_b.py")], cwd=K / "e15", env=ENV, capture_output=True,
                   text=True, timeout=3600)
(OUT / "test_duck_b.log").write_text(r.stdout + r.stderr, encoding="utf-8")
_l = (r.stdout + r.stderr).strip().splitlines()
REPORT["unit"] = {"rc": r.returncode, "sec": round(time.time() - t), "summary": _l[-1] if _l else "",
                  "passes": sum(1 for x in _l if x.startswith("PASS")), "fails": [x for x in _l if x.startswith("FAIL")][:40],
                  "skips": [x for x in _l if x.startswith("SKIP")][:40]}
save()
print("unit:", REPORT["unit"]["summary"], "| PASS", REPORT["unit"]["passes"], "| rc", r.returncode, flush=True)
for label, nbk, mode, fault in RUNS:
    env = dict(ENV, E14_HARNESS_ROOT=f"/tmp/e15b_h_{label}", E14S_MODE=mode)
    env.pop("E15B_FAULT", None)
    if fault:
        env["E15B_FAULT"] = fault
    t = time.time()
    try:
        r = subprocess.run([sys.executable, str(K / "e15/harness/harness_e15b_nb.py"), str(K / "nb" / NBS[nbk]), "0.03"],
                           cwd=K / "e15/harness", env=env, capture_output=True, text=True, timeout=5400)
        out, rc = r.stdout + r.stderr, r.returncode
    except subprocess.TimeoutExpired as e:
        out, rc = (e.stdout or "") if isinstance(e.stdout, str) else "", "timeout"
    (OUT / f"harness_{label}.log").write_text(out, encoding="utf-8")
    lines = out.splitlines()
    REPORT["harness"][label] = {"rc": rc, "sec": round(time.time() - t), "ok": "@@NB_OK@@" in out,
                                "summary": [x for x in lines if x.startswith("[e15b")],
                                "fails": [x[:1500] for x in lines if x.startswith("   FAIL")],
                                "traceback_tail": out[-3000:] if "Traceback" in out and "@@NB_OK@@" not in out else ""}
    save()
    print(label, "->", REPORT["harness"][label]["summary"], "ok" if REPORT["harness"][label]["ok"] else "FAIL",
          REPORT["harness"][label]["fails"][:6], flush=True)
REPORT["sec"] = round(time.time() - T0)
REPORT["all_ok"] = REPORT["unit"]["rc"] == 0 and all(v["ok"] for v in REPORT["harness"].values())
save()
print("ALL OK" if REPORT["all_ok"] else "NOT OK", "| total", REPORT["sec"], "s", flush=True)
